# nb165 v2 — Robust Boltz2 on Kaggle GPU (P100/T4 aware)

Robust patches:
1. **GPU detection in cell 1 only**, then all torch ops via subprocess (avoid in-process reload crashes)
2. **Conditional downgrade**: P100 (cc<7) gets torch 2.4.0, T4 keeps system torch
3. **Subprocess CUDA validation** in fresh Python to detect kernel-image mismatches early
4. **Fallback flag combinations** for boltz predict: try aggressive first, fall back to minimal if it errors
5. **Checkpoint writes** at every step — no silent failure mode
6. **Wrapped subprocesses** with explicit env and PYTHONNOUSERSITE to avoid stale imports

In [ ]:
# Cell 1: GPU detection (in-process torch) — must be FIRST so torch isn't yet reinstalled
import os, sys, time, subprocess, json
from pathlib import Path

TRACE_PATH = '/kaggle/working/trace.log'
CHECKPOINT_PATH = '/kaggle/working/checkpoint.json'

def W(msg):
    line = f'[{time.strftime("%H:%M:%S")}] {msg}\n'
    with open(TRACE_PATH, 'a') as f: f.write(line)
    print(msg, flush=True)

def checkpoint(stage, **kw):
    d = {}
    if Path(CHECKPOINT_PATH).exists():
        try: d = json.loads(open(CHECKPOINT_PATH).read())
        except Exception: d = {}
    d[stage] = {'time': time.strftime('%H:%M:%S'), **kw}
    Path(CHECKPOINT_PATH).write_text(json.dumps(d, indent=2))

W('=== nb165 v2 START ===')
W(f'python={sys.version[:40]}')

import torch
cuda_ok = torch.cuda.is_available()
cc = torch.cuda.get_device_capability(0) if cuda_ok else (0, 0)
device_name = torch.cuda.get_device_name(0) if cuda_ok else 'none'
W(f'system torch={torch.__version__} cuda={cuda_ok} cc={cc} device={device_name}')

is_p100 = cc[0] < 7
is_t4   = cc == (7, 5)
need_downgrade = is_p100
W(f'is_p100={is_p100} is_t4={is_t4} need_downgrade={need_downgrade}')

checkpoint('cell1_gpu_detect', torch_ver=torch.__version__, cc=list(cc),
           device=device_name, need_downgrade=need_downgrade)
W('=== CELL 1 OK ===')

In [ ]:
# Cell 2: conditional torch downgrade + boltz install (subprocess, no in-process imports)
W('=== CELL 2: INSTALL ===')
checkpoint('cell2_start')

if need_downgrade:
    W('Installing torch 2.4.0+cu121 for P100 compatibility...')
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                        'torch==2.4.0', 'torchvision==0.19.0',
                        '--index-url', 'https://download.pytorch.org/whl/cu121'],
                       capture_output=True, text=True, timeout=1200)
    W(f'torch downgrade rc={r.returncode} elapsed={time.time()-t0:.0f}s')
    if r.returncode != 0:
        W(f'  stderr last 2000: {r.stderr[-2000:]}')
    checkpoint('cell2_torch_downgrade', rc=r.returncode, elapsed=time.time()-t0)
else:
    W('Skip torch downgrade (T4 or no GPU)')
    checkpoint('cell2_torch_downgrade', skipped=True)

# Install boltz (transformers + lightning come with it — best installed AFTER torch downgrade)
W('Installing boltz...')
t0 = time.time()
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'boltz'],
                   capture_output=True, text=True, timeout=1200)
W(f'boltz install rc={r.returncode} elapsed={time.time()-t0:.0f}s')
if r.returncode != 0:
    W(f'  stderr last 2000: {r.stderr[-2000:]}')
checkpoint('cell2_boltz_install', rc=r.returncode, elapsed=time.time()-t0)
W('=== CELL 2 OK ===')

In [ ]:
# Cell 3: validate torch + cuda via SUBPROCESS only (do NOT touch in-process torch)
W('=== CELL 3: TORCH SUBPROCESS VALIDATION ===')
env_clean = {**os.environ, 'PYTHONNOUSERSITE': '1'}

validate_code = (
    'import torch, sys;'
    'print(f"PY_PATH={sys.executable}");'
    'print(f"TORCH_VER={torch.__version__}");'
    'print(f"TORCH_FILE={torch.__file__}");'
    'print(f"CUDA_AVAIL={torch.cuda.is_available()}");'
    'cc=torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0);'
    'print(f"CC={cc}");'
    'x=torch.zeros(1, device="cuda");'
    'print(f"ALLOC_OK={x}");'
    'y=x+1;'
    'print(f"ARITH_OK={y}")'
)
r = subprocess.run([sys.executable, '-c', validate_code],
                   env=env_clean, capture_output=True, text=True, timeout=120)
W(f'validation rc={r.returncode}')
W(f'  stdout: {r.stdout}')
if r.returncode != 0:
    W(f'  stderr last 2000: {r.stderr[-2000:]}')
checkpoint('cell3_validation', rc=r.returncode, stdout=r.stdout)
torch_works = r.returncode == 0 and 'ALLOC_OK' in r.stdout
W(f'torch_works={torch_works}')
W('=== CELL 3 OK ===')

In [ ]:
# Cell 4: boltz --help via subprocess
W('=== CELL 4: boltz --help ===')
r = subprocess.run(['boltz', '--help'], env=env_clean, capture_output=True, text=True, timeout=120)
W(f'boltz --help rc={r.returncode}')
W(f'  stdout first 600: {r.stdout[:600]}')
if r.returncode != 0:
    W(f'  stderr last 2000: {r.stderr[-2000:]}')
checkpoint('cell4_help', rc=r.returncode)
boltz_works = r.returncode == 0
W(f'boltz_works={boltz_works}')
W('=== CELL 4 OK ===')

In [ ]:
# Cell 5: predict with fallback flag combinations
W('=== CELL 5: PREDICT ===')
PXR_SEQ = 'LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCSIVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWEVLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELNGLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWANKTKDPLLLEAHALDQFSCK'
yaml_str = (
    f'version: 1\n'
    f'sequences:\n'
    f'- protein:\n    id: A\n    sequence: {PXR_SEQ}\n'
    f'- ligand:\n    id: B\n    smiles: CCOc1ccccc1\n'
    f'properties:\n- affinity:\n    binder: B\n'
)
Path('/kaggle/working/test.yaml').write_text(yaml_str)
OUT = Path('/kaggle/working/o_test')
OUT.mkdir(exist_ok=True)

# Try 3 flag combinations: aggressive (fast) -> medium -> minimal (most likely to succeed)
flag_combos = [
    {'name': 'fast', 'flags': ['--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']},
    {'name': 'minimal', 'flags': ['--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '20']},
    {'name': 'cpu_fallback', 'flags': ['--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '20', '--accelerator', 'cpu']},
]

success = False
for combo in flag_combos:
    out_dir = Path(f'/kaggle/working/o_{combo["name"]}')
    out_dir.mkdir(exist_ok=True)
    cmd = ['boltz', 'predict', '/kaggle/working/test.yaml', '--out_dir', str(out_dir), '--use_msa_server'] + combo['flags']
    W(f'TRY {combo["name"]}: {" ".join(cmd)}')
    t0 = time.time()
    try:
        r = subprocess.run(cmd, env=env_clean, capture_output=True, text=True, timeout=3600)
        elapsed = (time.time() - t0) / 60
        W(f'  rc={r.returncode} elapsed={elapsed:.1f}min')
        checkpoint(f'cell5_predict_{combo["name"]}', rc=r.returncode, elapsed_min=elapsed)
        if r.returncode == 0:
            W(f'  stdout tail 1500: {r.stdout[-1500:]}')
            # Check for affinity JSON
            aff_files = list(out_dir.rglob('*affinity*.json'))
            W(f'  affinity files: {len(aff_files)}')
            for jf in aff_files:
                W(f'  AFF {jf.name}: {open(jf).read()[:500]}')
            if aff_files:
                success = True
                W(f'=== SUCCESS WITH {combo["name"]} ===')
                checkpoint('final', success=True, winning_combo=combo['name'])
                break
        else:
            W(f'  stderr tail 2000: {r.stderr[-2000:]}')
    except subprocess.TimeoutExpired:
        W(f'  TIMEOUT after 60min')
        checkpoint(f'cell5_predict_{combo["name"]}', timeout=True)
    except Exception as e:
        W(f'  EXCEPTION: {type(e).__name__}: {e}')
        checkpoint(f'cell5_predict_{combo["name"]}', exception=str(e))

if not success:
    checkpoint('final', success=False)
    W('=== ALL COMBOS FAILED ===')
W('=== nb165 v2 END ===')